In [ ]:
!pip install --upgrade --no-cache-dir gdown
!gdown 1QTi7dJtNAfFR5mG0rd8K3ZGvEIfSn_DS
!unzip PersianData.zip

Downloading...
From: https://drive.google.com/uc?id=1QTi7dJtNAfFR5mG0rd8K3ZGvEIfSn_DS
To: /content/PersianData.zip
100% 7.46k/7.46k [00:00<00:00, 17.4MB/s]
Archive:  PersianData.zip
  inflating: 1.jpg                   
  inflating: 2.jpg                   
  inflating: 3.jpg                   
  inflating: 4.jpg                   
  inflating: 5.jpg                   


# my noise

In [ ]:
from PIL import Image, ImageDraw
import random
import numpy as np

def convertImageToBinary(path):
    """
    Convert an image to a binary representation based on pixel intensity.

    Args:
        path (str): The file path to the input image.

    Returns:
        list: A binary representation of the image where white is represented by -1 and black is represented by 1.
    """
    # Open the image file.
    image = Image.open(path)

    # Create a drawing tool for manipulating the image.
    draw = ImageDraw.Draw(image)

    # Determine the image's width and height in pixels.
    width = image.size[0]
    height = image.size[1]

    # Load pixel values for the image.
    pix = image.load()

    # Define a factor for intensity thresholding.
    factor = 100

    # Initialize an empty list to store the binary representation.
    binary_representation = np.zeros((height, width), dtype=int)

    # Loop through all pixels in the image.
    for i in range(width):
        for j in range(height):
            # Extract the Red, Green, and Blue (RGB) values of the pixel.
            red = pix[i, j][0]
            green = pix[i, j][1]
            blue = pix[i, j][2]

            # Calculate the total intensity of the pixel.
            total_intensity = red + green + blue

            # Determine whether the pixel should be white or black based on the intensity.
            if total_intensity > (((255 + factor) // 2) * 3):
                red, green, blue = 255, 255, 255  # White pixel
                binary_representation [j,i] = -1
            else:
                red, green, blue = 0, 0, 0  # Black pixel
                binary_representation [j,i] = 1

            # Set the pixel color accordingly.
            draw.point((i, j), (red, green, blue))

    # Clean up the drawing tool.
    del draw

    # Return the binary representation of the image.
    return binary_representation

# New


In [ ]:
def to_vector(matrix):
    return matrix.flatten().reshape(-1, 1)

In [ ]:
def get_noisy_binary_matrix_from_image(image_path, noise_factor):
    """
    میاد روی تصویر ورودی نویز RGB اضافه می‌کنه
    بعدش هم تصویر رو به باینری تبدیل می‌کنه و برمی‌گردونه
    """
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    pix = image.load()

    for i in range(width):
        for j in range(height):
            rand = random.randint(-noise_factor, noise_factor)

            # Add the noise to the Red, Green, and Blue (RGB) values of the pixel.
            red = pix[i, j][0] + rand
            green = pix[i, j][1] + rand
            blue = pix[i, j][2] + rand

            # Ensure that RGB values stay within the valid range (0-255).
            if red < 0:
                red = 0
            if green < 0:
                green = 0
            if blue < 0:
                blue = 0
            if red > 255:
                red = 255
            if green > 255:
                green = 255
            if blue > 255:
                blue = 255
            draw.point((i, j), (red, green, blue))
    image.save('/content/noisy2.jpg')
    # بعد از اعمال نویز، تبدیل به ماتریس باینری
    binary_matrix = convertImageToBinary('2.jpg')
    for i in range(width):
        for j in range(height):
            red = pix[i, j][0]
            green = pix[i, j][1]
            blue = pix[i, j][2]
            total = red + green + blue
            binary_matrix[j, i] = -1 if total > ((255 + 100) // 2) * 3 else 1

    del draw
    return binary_matrix

In [ ]:
def hopfield_train(patterns):
    n = patterns[0].shape[0]
    W = np.zeros((n, n))
    for p in patterns:
        W += p @ p.T
    np.fill_diagonal(W, 0)
    return W

In [ ]:
# بازیابی الگو از نسخه‌ی نویزی
def hopfield_recall(W, input_vector, max_iters=10):
    x = input_vector.copy()
    for _ in range(max_iters):
        x_new = np.sign(W @ x)  # به‌روز رسانی
        x_new[x_new == 0] = 1   # اگر صفر شد، بکن 1
        if np.array_equal(x, x_new):
            break  # اگر ثابت شد، دیگه ادامه نده
        x = x_new
    return x

In [ ]:
#  ذخیره‌ی خروجی
def to_image(matrix, path):
    h, w = matrix.shape
    image = Image.new("RGB", (w, h), "white")
    draw = ImageDraw.Draw(image)
    for i in range(h):
        for j in range(w):
            val = 255 if matrix[i, j] == -1 else 0

            draw.point((j, i), (val, val, val))

    image.save(path)


In [ ]:

path = '2.jpg'

#  ساخت بردار آموزشی
clean_matrix = convertImageToBinary('2.jpg')
clean_vector = to_vector(clean_matrix)

# آموزش شبکه
W = hopfield_train([clean_vector])

# ساخت ورودی نویزی با تابع خودت
noisy_matrix = get_noisy_binary_matrix_from_image(path, noise_factor=100000)
noisy_vector = to_vector(noisy_matrix)

# بازیابی با شبکه هاپفیلد
recovered_vector = hopfield_recall(W, noisy_vector)

# تبدیل خروجی به تصویر
recovered_matrix = recovered_vector.reshape(clean_matrix.shape)
to_image(recovered_matrix, 'recovered_from_noisy2.jpg')